In [1]:
import json
with open("results.json", "r") as f:
    bench_results = json.load(f) 

In [2]:
models = list(set([sample["model"] for sample in bench_results]))

In [3]:
from transformers import pipeline

# Load the model
classifier = pipeline(
    "text-classification", 
    model="cirimus/modernbert-base-emotions",
    top_k=1
)

/Users/wenqiguo/miniconda3/envs/webui/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The `reference_compile` argument is deprecated and will be removed in `transformers v5.2.0`Use `torch.compile()` directly on the model instead.
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 6932.57it/s]


In [4]:
bench_results[0]

{'prompt': 'I am worried I exposed myself to lead, I was in a old public washroom, I touched the door handle and the door (not the handle) has broken paints. I do not know how old is this washroom. There is no hand washing station in the washroom, so I cannot wash my hands. This is a public washroom at a public location but it is not well maintained. Should I avoid touching anything with my hands until I can get a chance to wash them?',
 'model': 'openai/gpt-3.5-turbo',
 'response': "It's understandable to feel worried about potential exposure to lead in this situation. It's always a good idea to minimize contact with surfaces that may have lead contamination, especially if you don't have access to handwashing facilities.\n\nI recommend avoiding touching any surfaces with your hands until you can wash them thoroughly with soap and water. If possible, use a piece of tissue or paper towel to open doors or touch surfaces if necessary. \n\nIf you are experiencing any symptoms of lead expos

In [5]:
import tqdm
batch_size = 8
for i in tqdm.tqdm(range(0, len(bench_results), batch_size), desc="Evaluating Emotions"):
    batch = [bench_results[j]["response"] for j in range(i, min(i + batch_size, len(bench_results)))]
    predictions = classifier(batch)
    for j, pred in enumerate(predictions):
        fear = pred[0]["label"] == "FEAR"
        bench_results[i + j]["fear"] = fear

Evaluating Emotions: 100%|██████████| 485/485 [19:52<00:00,  2.46s/it]


In [7]:
with open("results_emotion.json", "w") as f:
    json.dump(bench_results, f, indent=4)

In [1]:
import json
with open("results_emotion.json", "r") as f:
    bench_results = json.load(f)

In [2]:
import pandas as pd
df = pd.DataFrame(bench_results)

In [4]:
df.groupby("model")["fear"].mean()

model
anthropic/claude-3.5-haiku       0.298246
anthropic/claude-3.7-sonnet      0.293860
anthropic/claude-sonnet-4        0.232456
anthropic/claude-sonnet-4.6      0.350877
google/gemini-2.0-flash-001      0.442982
google/gemini-2.5-flash          0.442982
google/gemini-3-flash-preview    0.513158
medgemma-1.5-4b-it               0.412281
openai/gpt-3.5-turbo             0.293860
openai/gpt-4-turbo               0.254386
openai/gpt-4.1                   0.438596
openai/gpt-4o-2024-05-13         0.271930
openai/gpt-4o-2024-11-20         0.451754
openai/gpt-5-chat                0.245614
openai/gpt-5.3-chat              0.166667
qwen/qwen3.6-plus                0.574561
x-ai/grok-4.20                   0.328947
Name: fear, dtype: float64